In [22]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sqlalchemy import create_engine

pio.renderers.default = "notebook"

In [4]:
BASE = "/Users/mojo/Documents/MyProjects/Neighborhood_Appreciation_Tracker/"

df_2025 = pd.read_csv(
    f'{BASE}data/real_acct_2025.txt',
    sep='\t',
    low_memory=False,
    on_bad_lines='skip',
    encoding='latin-1'
)

df_2025.shape

(1601831, 71)

In [9]:
print(df_2025.head(3))

          acct    yr                      mailto  \
0  10010000013  2025             CITY OF HOUSTON   
1  10020000001  2025               CURRENT OWNER   
2  10020000003  2025  MILBY CHARLES FAMILY PTNSH   

                      mail_addr_1 mail_addr_2 mail_city mail_state  \
0                     PO BOX 1562               HOUSTON         TX   
1  1717 SAINT JAMES PLACE STE 112               HOUSTON         TX   
2                2612 TODVILLE RD              SEABROOK         TX   

     mail_zip mail_country undeliverable  ... protested  certified_date  \
0  77251-1562                          N  ...         N      08/22/2025   
1  77056-3412                          N  ...         Y      10/10/2025   
2  77586-3008                          N  ...         Y      07/25/2025   

       rev_dt rev_by  new_own_dt                        lgl_1 lgl_2 lgl_3  \
0  01/01/2016  01391  01/02/1988                    ALL BLK 1  SSBB   NaN   
1  03/08/2023  01710  10/31/2012                  TR 15

In [6]:
df_2025.columns.tolist()

['acct',
 'yr',
 'mailto',
 'mail_addr_1',
 'mail_addr_2',
 'mail_city',
 'mail_state',
 'mail_zip',
 'mail_country',
 'undeliverable',
 'str_pfx',
 'str_num',
 'str_num_sfx',
 'str',
 'str_sfx',
 'str_sfx_dir',
 'str_unit',
 'site_addr_1',
 'site_addr_2',
 'site_addr_3',
 'state_class',
 'school_dist',
 'map_facet',
 'key_map',
 'Neighborhood_Code',
 'Neighborhood_Grp',
 'Market_Area_1',
 'Market_Area_1_Dscr',
 'Market_Area_2',
 'Market_Area_2_Dscr',
 'econ_area',
 'econ_bld_class',
 'center_code',
 'yr_impr',
 'yr_annexed',
 'splt_dt',
 'dsc_cd',
 'nxt_bld',
 'bld_ar',
 'land_ar',
 'acreage',
 'Cap_acct',
 'shared_cad',
 'land_val',
 'bld_val',
 'x_features_val',
 'ag_val',
 'assessed_val',
 'tot_appr_val',
 'tot_mkt_val',
 'prior_land_val',
 'prior_bld_val',
 'prior_x_features_val',
 'prior_ag_val',
 'prior_tot_appr_val',
 'prior_tot_mkt_val',
 'new_construction_val',
 'tot_rcn_val',
 'value_status',
 'noticed',
 'notice_dt',
 'protested',
 'certified_date',
 'rev_dt',
 'rev_by',
 '

In [7]:
cols = ['acct', 'yr', 'site_addr_3', 'tot_appr_val', 'prior_tot_appr_val', 'land_val', 'bld_val', 'state_class', 'Neighborhood_Code']
df_2025[cols].isnull().sum()

acct                      0
yr                        0
site_addr_3               0
tot_appr_val            876
prior_tot_appr_val    28055
land_val                876
bld_val                 876
state_class               0
Neighborhood_Code         0
dtype: int64

In [8]:
df_2025['state_class'].value_counts().head(10)

state_class
A1    1142786
X1      77038
F1      67865
C1      59727
C3      36856
M3      31678
C2      27988
Z4      27223
O1      15143
Z3      14010
Name: count, dtype: int64

In [10]:
df_2025['site_addr_3'].value_counts().head(10)

site_addr_3
77433    50312
77449    42977
77429    34546
77493    31935
77084    30917
77379    29671
77346    28396
77375    27751
77373    27312
77521    22668
Name: count, dtype: int64

In [11]:
residential = df_2025[df_2025['state_class'] == 'A1']

residential['tot_appr_val'].describe()

count    1.142574e+06
mean     3.687192e+05
std      4.103113e+05
min      2.300000e+01
25%      2.115090e+05
50%      2.789865e+05
75%      3.872888e+05
max      3.354664e+07
Name: tot_appr_val, dtype: float64

In [12]:
years = [2022, 2023, 2024, 2025]
dfs = {}

for yr in years:
    dfs[yr] = pd.read_csv(
        f'{BASE}data/real_acct_{yr}.txt',
        sep='\t',
        low_memory=False,
        on_bad_lines='skip',
        encoding='latin-1'
    )
    print(f"{yr}: {dfs[yr].shape}")

2022: (1531477, 71)
2023: (1555743, 71)
2024: (1582018, 71)
2025: (1601831, 71)


In [13]:
cols = [
    'acct', 'yr', 'site_addr_3', 'tot_appr_val',
    'prior_tot_appr_val', 'land_val', 'bld_val',
    'state_class', 'Neighborhood_Code'
]

frames = []
for yr, df in dfs.items():
    residential = df[df['state_class'] == 'A1'][cols].copy()
    frames.append(residential)

combined = pd.concat(frames, ignore_index=True)
combined.shape

(4477497, 9)

In [15]:
combined = combined[combined['tot_appr_val'] > 1000]
combined['site_addr_3'] = combined['site_addr_3'].str.strip()
combined['acct'] = combined['acct'].astype(str).str.strip()
combined.isnull().sum()

acct                      0
yr                        0
site_addr_3               0
tot_appr_val              0
prior_tot_appr_val    13618
land_val                  0
bld_val                   0
state_class               0
Neighborhood_Code         0
dtype: int64

In [16]:
engine = create_engine(f'sqlite:///{BASE}data/houston_re.db')

combined.to_sql('property_values', engine, if_exists='replace', index=False)

print("Done")

Done


In [17]:
neighborhoods = pd.read_csv(
    f'{BASE}data/real_neighborhood_code.txt',
    sep='\t',
    encoding='latin-1'
)

neighborhoods.to_sql('neighborhood_codes', engine, if_exists='replace', index=False)

print(neighborhoods.shape)

(11044, 3)


In [18]:
query = """
SELECT
    site_addr_3 AS zip_code,
    yr,
    ROUND(AVG(tot_appr_val), 2) AS avg_appr_val,
    COUNT(*) AS property_count
FROM property_values
WHERE tot_appr_val > 0
    AND site_addr_3 IS NOT NULL
GROUP BY site_addr_3, yr
ORDER BY zip_code, yr
"""

df_zip_yr = pd.read_sql(query, engine)
df_zip_yr.to_csv(f'{BASE}data/zip_year_avg_values.csv', index=False)
print(f"Saved {len(df_zip_yr)} rows")

Saved 581 rows


In [19]:
query_yoy = """
WITH zip_yearly AS (
    SELECT
        site_addr_3 AS zip_code,
        yr,
        ROUND(AVG(tot_appr_val), 2) AS avg_appr_val,
        COUNT(*) AS property_count
    FROM property_values
    WHERE tot_appr_val > 0
        AND site_addr_3 IS NOT NULL
        AND site_addr_3 != ''
    GROUP BY site_addr_3, yr
),
with_lag AS (
    SELECT
        zip_code,
        yr,
        avg_appr_val,
        LAG(avg_appr_val) OVER (PARTITION BY zip_code ORDER BY yr) AS prev_year_val
    FROM zip_yearly
)
SELECT
    zip_code,
    yr,
    avg_appr_val,
    ROUND(((avg_appr_val / prev_year_val) - 1) * 100, 2) AS yoy_change_pct
FROM with_lag
WHERE prev_year_val IS NOT NULL
    AND zip_code != ''
ORDER BY zip_code, yr
"""

df_yoy = pd.read_sql(query_yoy, engine)
df_yoy.to_csv(f'{BASE}data/yoy_appreciation.csv', index=False)
print(f"Saved {len(df_yoy)} rows")

Saved 432 rows


In [20]:
query_cagr = """
WITH zip_yearly AS (
    SELECT
        site_addr_3 AS zip_code,
        yr,
        ROUND(AVG(tot_appr_val), 2) AS avg_appr_val,
        COUNT(*) AS property_count
    FROM property_values
    WHERE tot_appr_val > 0
        AND site_addr_3 IS NOT NULL
    GROUP BY site_addr_3, yr
),
cagr_pivot AS (
    SELECT
        zip_code,
        MAX(CASE WHEN yr = 2022 THEN avg_appr_val END) AS val_2022,
        MAX(CASE WHEN yr = 2025 THEN avg_appr_val END) AS val_2025
    FROM zip_yearly
    GROUP BY zip_code
)
SELECT
    zip_code,
    val_2022,
    val_2025,
    ROUND((POWER(val_2025 / val_2022, 1.0/3) - 1) * 100, 2) AS cagr_pct
FROM cagr_pivot
WHERE val_2022 IS NOT NULL
    AND val_2025 IS NOT NULL
ORDER BY cagr_pct DESC, zip_code
"""

df_cagr = pd.read_sql(query_cagr, engine)
df_cagr.to_csv(f'{BASE}data/cagr_by_zip.csv', index=False)
print(f"Saved {len(df_cagr)} rows")

Saved 144 rows


In [21]:
query_ranked = """
WITH zip_yearly AS (
    SELECT
    site_addr_3 AS zip_code,
    yr,
    ROUND(AVG(tot_appr_val), 2) AS avg_appr_val,
    COUNT(*) AS property_count
FROM property_values
WHERE tot_appr_val > 0
    AND site_addr_3 IS NOT NULL
GROUP BY site_addr_3, yr
),
cagr_pivot AS (
    SELECT
        zip_code,
        MAX(CASE WHEN yr = 2022 THEN avg_appr_val END) AS val_2022,
        MAX(CASE WHEN yr = 2025 THEN avg_appr_val END) AS val_2025
    FROM zip_yearly
    GROUP BY zip_code
),
cagr_final AS(
	SELECT
		zip_code,
		val_2022,
		val_2025,
		ROUND((POWER(val_2025 / val_2022, 1.0/3) - 1) * 100, 2) AS cagr_pct
	FROM cagr_pivot
	WHERE val_2022 IS NOT NULL
		AND val_2025 IS NOT NULL
),
ranked AS(
    SELECT
        zip_code,
		val_2022,
		val_2025,
		cagr_pct,
        RANK() OVER (ORDER BY cagr_pct DESC) AS rank_top,
        RANK() OVER (ORDER BY cagr_pct ASC) AS rank_bottom
    FROM cagr_final
)
SELECT
    zip_code,
    val_2022,
    val_2025,
    cagr_pct,
	    CASE WHEN rank_top <= 10 THEN 'Top 10'
         WHEN rank_bottom <= 10 THEN 'Bottom 10'
    END AS tier
FROM ranked
WHERE rank_top <= 10 OR rank_bottom <= 10
ORDER BY cagr_pct DESC;
"""

df_ranked = pd.read_sql(query_ranked, engine)
df_ranked.to_csv(f'{BASE}data/top_bottom_zips.csv', index=False)
print(df_ranked)

   zip_code    val_2022    val_2025  cagr_pct       tier
0     77535   136921.86   224928.71     17.99     Top 10
1     77357   186815.79   290642.12     15.87     Top 10
2     77028   103695.34   149531.66     12.98     Top 10
3     77026   110624.71   153882.88     11.63     Top 10
4     77093   123345.28   167631.01     10.77     Top 10
5     77039   142033.25   191718.62     10.52     Top 10
6     77091   190273.47   256724.93     10.50     Top 10
7     77011   157530.34   211272.19     10.28     Top 10
8     77053   145532.00   194429.38     10.14     Top 10
9     77016   116370.44   154923.41     10.01     Top 10
10    77098   811071.41   921282.77      4.34  Bottom 10
11    77035   244925.30   277878.52      4.30  Bottom 10
12    77081   417604.90   471732.25      4.15  Bottom 10
13    77057   768086.17   861102.19      3.88  Bottom 10
14    77063   472348.40   527898.64      3.78  Bottom 10
15    77056  1270541.08  1407363.01      3.47  Bottom 10
16    77365   379078.12   41273

In [29]:
df_ranked = df_ranked.sort_values('cagr_pct', ascending=True)

fig = px.bar(
    df_ranked,
    x="cagr_pct",
    y="zip_code",
    color="tier",
    orientation='h',
    title='Top and Bottom Zip Codes by CAGR',
    text="cagr_pct"
)

fig.update_layout(
    xaxis_title="CAGR (%)",
    yaxis_title="Zip Code",
    legend_title="Tier",
    yaxis={'categoryorder':'total ascending'}
)

fig.write_image(f'{BASE}output/cagr_bar_chart.png')
fig.show()

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
